In [2]:
import pandas as pd
import glob

# Busca los archivos csv que empiecen por limpieza y terminen en .csv
archivos_csv = glob.glob("super*.csv")

# Verificar que se encontraron archivos
if len(archivos_csv) == 0:
    print("No se encontraron archivos que empiecen por 'limpieza'")
    print("Archivos en la carpeta actual:")
    import os
    for archivo in os.listdir('.'):
        if archivo.endswith('.csv'):
            print(f"   - {archivo}")
    exit()

print(f"Encontrados {len(archivos_csv)} archivos:")
for archivo in archivos_csv:
    print(f"   - {archivo}")

# Leer todos los archivos encontrados
dataframes = []
for archivo in archivos_csv:
    print(f"Leyendo: {archivo}")
    df = pd.read_csv(archivo, sep=';')
    print(f"   → {len(df)} registros, {len(df.columns)} columnas")
    dataframes.append(df)

# Verificar que tenemos exactamente 3 archivos (demanda, producción, precios)
if len(dataframes) != 3:
    print(f"Se esperaban 3 archivos, pero se encontraron {len(dataframes)}")
    print("   Los archivos deberían ser: limpieza_demanda.csv, limpieza_produccion.csv, limpieza_precios.csv")
    # Intentar continuar con los que se encontraron

# Unificar los datasets usando merge por datetime
print("\n Unificando datasets...")

# Empezar con el primer dataframe
df_unificado = dataframes[0]

# Unir con los siguientes dataframes uno por uno
for i in range(1, len(dataframes)):
    df_unificado = pd.merge(df_unificado, dataframes[i], 
                           on='datetime', 
                           how='inner')
    print(f"   Unión {i}: {len(df_unificado)} registros")

# Limpiar columnas duplicadas de 'hour'
columnas_hour = [col for col in df_unificado.columns if col == 'hour' or col.startswith('hour_')]
if len(columnas_hour) > 1:
    print(f"\n Eliminando columnas 'hour' duplicadas...")
    # Conservar solo la primera columna 'hour'
    if 'hour' in df_unificado.columns:
        for col in columnas_hour:
            if col != 'hour':
                df_unificado.drop(columns=[col], inplace=True)
    print(f"   Columnas eliminadas: {len(columnas_hour)-1}")

# Ordenar por fecha
if 'datetime' in df_unificado.columns:
    print("\n Ordenando por fecha...")
    df_unificado['datetime'] = pd.to_datetime(df_unificado['datetime'])
    df_unificado = df_unificado.sort_values('datetime').reset_index(drop=True)
    print(f"   Rango de fechas: {df_unificado['datetime'].min()} a {df_unificado['datetime'].max()}")

# Guardar el dataset unificado
nombre_archivo = "Dataset_Unificado1.csv"
df_unificado.to_csv(nombre_archivo, sep=';', index=False)

# Mostrar resultados finales
print("\n" + "="*60)
print(" DATASET UNIFICADO COMPLETADO")
print("="*60)
print(f" Archivos procesados: {len(archivos_csv)}")
for archivo in archivos_csv:
    print(f"   - {archivo}")
print(f"\n Registros totales: {len(df_unificado):,}")
print(f" Columnas totales: {len(df_unificado.columns)}")
print(f" Archivo guardado: {nombre_archivo}")

print("\n Primeras 3 filas:")
print(df_unificado.head(3))

print("\n Columnas disponibles:")
for i, col in enumerate(df_unificado.columns, 1):
    print(f"   {i:2d}. {col}")

Encontrados 2 archivos:
   - superDataset_Unificado1.csv
   - super_dataset_climatologia.csv
Leyendo: superDataset_Unificado1.csv
   → 43848 registros, 22 columnas
Leyendo: super_dataset_climatologia.csv
   → 43848 registros, 13 columnas
Se esperaban 3 archivos, pero se encontraron 2
   Los archivos deberían ser: limpieza_demanda.csv, limpieza_produccion.csv, limpieza_precios.csv

 Unificando datasets...
   Unión 1: 43848 registros

 Ordenando por fecha...
   Rango de fechas: 2020-01-01 00:00:00 a 2024-12-31 23:00:00

 DATASET UNIFICADO COMPLETADO
 Archivos procesados: 2
   - superDataset_Unificado1.csv
   - super_dataset_climatologia.csv

 Registros totales: 43,848
 Columnas totales: 34
 Archivo guardado: Dataset_Unificado1.csv

 Primeras 3 filas:
             datetime    demand  year  month  day  dayofweek  is_weekend  \
0 2020-01-01 00:00:00  23001.00  2020      1    1          2           0   
1 2020-01-01 01:00:00  22253.67  2020      1    1          2           0   
2 2020-01-01 